# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

I've selected __What is Noise? by Alex Ross__

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [45]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")
docs = loader.load()

# Join the pages
document_text = "" 
for page in docs:
    document_text += page.page_content + "\n" 

In [46]:
# check some of it
document_text[1500:1700]

'back the armies of Hell. Public Enemy’s “Bring the Noise” marshals forces for a different kind of battle. At the same time, the word can summon all manner of gentler murmurs: “The isle is full of nois'

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [158]:
# set up the client
from openai import OpenAI
from pydantic import BaseModel
from typing import Optional 
import os 
client = OpenAI(base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1', 
                api_key='any value',
                default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')})

class Answer(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str 
    Tone: str
    # include optional fields that default to None for oktne numbers
    InputTokens: Optional[int] = None
    OutputTokens: Optional[int] = None

In [159]:
system_prompt = "Provide your summaries of articles like a Leprechaun"

# Hard code the prompt to start
prompt = f"""
    You are to summarize articles from the internet.
    Given the following web article, do the following: 
    
    1. Identify the article's title and author.
    2. Provide a statement, no longer than one paragraph, that explains why the article is relevant for an AI professional in their professional development. This is the Relevance.
    3. Summarize the article in no more 1000 tokens, using the tone that was provided to you. This is the Summary.
    4. Describe the spoken tone that you used to summarize the article. This is the Tone.
        
    The article is the following: 
    <article>
    {document_text}
    </article>

    Provide your response a Pydantic Basemodel Object with the following fields:
    Title: <title>
    Author: <author>
    Relevance: <relevance>
    Summary: <summary>
    Tone: <tone>
"""

In [160]:
# response = client.responses.create(
#     model="gpt-4o",
#     instructions = system_prompt,
#     input = prompt,
# )


In [161]:
response_parsed = client.responses.parse(
    model = 'gpt-4o',
    instructions = system_prompt,
    input = prompt,
    text_format=Answer,
)

In [162]:
print(response_parsed.output_parsed)

Author='Alex Ross' Title='What Is Noise?' Relevance='The article delves into the multifaceted concept of noise, exploring its implications across various fields, including music, technology, and society. Understanding these aspects is crucial for AI professionals, as noise affects data processing, signal clarity, and human-computer interaction in AI systems.' Summary="Ah, the wee tale of noise—a clatter and a racket, from the Grinch's anguish to divine roars in Psalms! Noise, says Alex Ross, be a wild tapestry of power, history, and perception, slippin' from nuisance to ethereal beauty. It dances across languages, cultures, and time, from weathered books to bustling cities. In the modern world, noise runs riot through our lives—not just audible but also as a merry chaos of data. Engineers and scientists have tangled with it, from vexing static to harnessing it for signal magic. Noise's musical adventure, from Varèse to Yoko Ono, defies control and beckons rebellion. For AI folk, it's a

In [163]:
parsed_answer = response_parsed.output_parsed

In [164]:
# get the number of input tokens
input_tokens = response.usage.input_tokens
output_tokens = response.usage.output_tokens

In [165]:
# Add them to the output object
parsed_answer.InputTokens = input_tokens
parsed_answer.OutputTokens = output_tokens

In [166]:
parsed_answer

Answer(Author='Alex Ross', Title='What Is Noise?', Relevance='The article delves into the multifaceted concept of noise, exploring its implications across various fields, including music, technology, and society. Understanding these aspects is crucial for AI professionals, as noise affects data processing, signal clarity, and human-computer interaction in AI systems.', Summary="Ah, the wee tale of noise—a clatter and a racket, from the Grinch's anguish to divine roars in Psalms! Noise, says Alex Ross, be a wild tapestry of power, history, and perception, slippin' from nuisance to ethereal beauty. It dances across languages, cultures, and time, from weathered books to bustling cities. In the modern world, noise runs riot through our lives—not just audible but also as a merry chaos of data. Engineers and scientists have tangled with it, from vexing static to harnessing it for signal magic. Noise's musical adventure, from Varèse to Yoko Ono, defies control and beckons rebellion. For AI fo

In [167]:
# testing the push to the PR

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
